# Plots and animations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/core/09_plots_and_animations.ipynb)

Every figure in minilink works the same way, from a trajectory plot to a cost-to-go, a Bode diagram or a 3-D animation:

1. **A plot builds a description** of the figure: panels of layers (lines, heatmaps, level sets, surfaces, arrows), plain data.
2. **A renderer draws it**: `"matplotlib"` (the default) or `"plotly"`.
3. **A record comes back**: a `PlotResult` for a plot, an `AnimationResult` for an animation. It holds the drawn figure, shows it again and saves it.

Every plot takes the same three keywords: `renderer=`, `show=` and `ax=`. This chapter runs them all, and is the place to try how a figure behaves in a notebook, a console, a script and Colab.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

In [ ]:
import pathlib
import tempfile

import numpy as np

from minilink import PID, Pendulum

pendulum = Pendulum()
pendulum.x0 = np.array([2.0, 0.0])
traj = pendulum.compute_trajectory(tf=6.0)

# A folder for the files this chapter saves
out = pathlib.Path(tempfile.mkdtemp())

## 1. A plot and its record

A plot is shown at the line that draws it and returns a `PlotResult`:

| Field / call | Meaning |
| --- | --- |
| `result.renderer` | `"matplotlib"`, `"plotly"` or `"graphviz"` |
| `result.figure` | The drawn figure: matplotlib's or Plotly's figure, Graphviz's graph |
| `result.axes` | Matplotlib's `Axes` of a one-panel figure, a list in panel order for several; `None` on the other renderers |
| `result.spec` | The description the figure was drawn from |
| `result.show()` | Show it again, where the session puts it |
| `result.save(file_name)` | Write it, in the format of the extension |

In [ ]:
result = pendulum.plot_trajectory(traj)

In [ ]:
result.renderer, len(result.axes), [panel.y_label for panel in result.spec.panels]

## 2. Saving

One way for every figure: the record's `.save()`. The extension picks the format: matplotlib writes `.png`, `.pdf`, `.svg` and any format it knows; Plotly an interactive `.html` page, or an image.

In [ ]:
result.save(out / "pendulum.pdf")

## 3. Two renderers

`renderer="plotly"` draws the same description with Plotly: zoom, pan and hover in the browser or the notebook.

In [ ]:
pendulum.plot_trajectory(traj, renderer="plotly")

## 4. One plot into another: `ax=`

`ax=` says where to draw. It takes a record, whose panels receive the new plot's layers, or matplotlib axes of your own. A plot drawn into a host is not shown by itself (`show=None` means "show unless `ax=` is given"); the host is shown when you ask, with `.show()`.

Two runs on one trajectory figure, panel by panel; a line in a colour the host already draws takes the next free one:

In [ ]:
pendulum.x0 = np.array([1.0, 0.0])
other = pendulum.compute_trajectory(tf=6.0)

first = pendulum.plot_trajectory(traj, show=False)
pendulum.plot_trajectory(other, ax=first)
first.show()

The same on Plotly: the host's renderer draws the overlay.

In [ ]:
first = pendulum.plot_trajectory(traj, renderer="plotly", show=False)
pendulum.plot_trajectory(other, ax=first)
first.show()

Your own matplotlib axes work too: a plot draws in full on fresh axes, and you keep drawing on them with matplotlib.

In [ ]:
import matplotlib.pyplot as plt

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4))
pendulum.plot_phase_plane(traj, ax=left)
pendulum.plot_pzmap(ax=right)
right.set_title("pole-zero map")
plt.show()

A host refuses what it cannot take, by name: a 3-D plot into a 2-D panel, another number of panels, a second colour bar.

## 5. Where a figure goes: `show=`

| `show=` | Meaning |
| --- | --- |
| `None` (default on plots) | Shown, unless the plot is drawn into `ax=` |
| `True` | Shown where the session puts it (table below) |
| `"inline"` / `"window"` | That place, refused by name where it cannot be |
| `False` | Built only: `.show()` or `.save()` later |

Where `show=True` puts a figure:

| Renderer | Script, console (IPython, Spyder) | Jupyter, VS Code | Colab |
| --- | --- | --- | --- |
| matplotlib | its window (a script waits until you close it) | inline under the cell (a widget with `%matplotlib widget`) | inline |
| plotly | a browser tab | inline | inline |
| meshcat | the browser viewer | inline | inline |
| pygame | its window | its window | refused |
| graphviz | the vector PDF in the system's viewer | inline SVG | inline SVG |

In a notebook every figure appears **at the line that draws it**, in code order, even in a long cell or a function. A change made after a shown plot needs `show=False` first:

In [ ]:
result = pendulum.plot_trajectory(traj, show=False)
result.axes[0].axhline(np.pi, color="black", linestyle=":")
result.show()

**Headless runs.** With the environment variable `MINILINK_HEADLESS=1`, no figure is shown at all, whatever `show` asks: the tests, the demo checks and CI set it. Plots still build and save.

## 6. The description itself

A record's `spec` is the figure as data, a `FigureSpec`: a grid of `Panel`s, each an ordered list of layers. `draw` hands it to a renderer; `renderer=None` builds the description alone.

| Layer | Draws |
| --- | --- |
| `Trace` | a line or a point set; `fill=True` closes and fills it |
| `RefLine`, `Note` | a reference line, a boxed text |
| `Heatmap`, `Contour`, `Surface` | values on a mesh: colours, level sets, a 3-D surface |
| `Arrows` | a vector field, arrows or streamlines |

In [ ]:
from minilink.graphical import Contour, FigureSpec, Heatmap, Panel, Trace, draw

x = np.linspace(-2.0, 2.0, 41)
y = np.linspace(-2.0, 2.0, 41)
# A quadratic V(θ, ω) = θ² + ω², V[j, i] at (θ = x[i], ω = y[j])
V = np.add.outer(y**2, x**2)

spec = FigureSpec(
    panels=(
        Panel(
            layers=(
                Heatmap(x, y, V, cmap="YlOrRd", colorbar="V"),
                Contour(x, y, V, levels=(1.0,), color="black", label="V = 1"),
                Trace(other.x[0], other.x[1], color="#1f77b4", label="run"),
            ),
            title="a level set and a run",
            x_label="θ",
            y_label="ω",
        ),
    )
)
draw(spec)

In [ ]:
draw(spec, renderer="plotly")

## 7. Animations

`animate` plays a trajectory and returns an `AnimationResult`: `figure` (the figure, or Meshcat's viewer), `animation` (the renderer's animation object), `.show()` and `.save()`. Every renderer but pygame plays a clip built once from the frames.

In [ ]:
clip = pendulum.animate(traj)

In [ ]:
clip.save(out / "pendulum.gif")

In [ ]:
pendulum.animate(traj, renderer="plotly")

Meshcat draws in 3-D. In a notebook the clip plays inline; elsewhere it opens the browser viewer. `.save("…html")` writes a self-contained page to share.

In [ ]:
pendulum.animate(traj, renderer="meshcat", show=False).save(out / "pendulum.html")

**A note on Meshcat.** Geometry that changes during a clip (force arrows, a planned horizon, sampled futures) plays as pages switched on and off, one page per run of frames with the same shape.

## 8. Block diagrams

`plot_diagram` draws a system's blocks and wires with Graphviz: an SVG under the cell in a notebook, the vector PDF in the system's viewer in a script or a console, where you can zoom without loss.

In [ ]:
loop = PID(Kp=5.0) >> Pendulum()
diagram = loop.plot_diagram()

In [ ]:
diagram.save(out / "loop.svg")

## 9. Summary

| You want | You write |
| --- | --- |
| a plot | `sys.plot_trajectory()`, `plant.plot_bode()`, `sol.plot_cost_to_go()`, … |
| it with Plotly | `renderer="plotly"` |
| it built, not shown | `show=False`, then `result.show()` |
| it in a file | `result.save("figure.pdf")`, `clip.save("clip.gif")`, `diagram.save("loop.svg")` |
| one plot into another | `ax=result`, or `ax=` your matplotlib axes |
| no figure at all (tests, CI) | `MINILINK_HEADLESS=1` |
| the figure as data | `result.spec`, `draw(spec, renderer=...)` |